In [62]:
import math
import numpy as np

In [63]:
class Value:
    def __init__(self, data, _children = (), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self._backward = lambda: None
        self.grad = 0

    def __repr__(self):
        return f"Value(data={self.data}, label={self.label}, gradient={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other.data)
        out = Value(self.data + other.data, _children = (self, other), _op='+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
        out._backward = _backward
        return out;

    def __radd__(self, other):
        return self + other;

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other.data)
        out = Value(self.data * other.data, _children = (self, other), _op='*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out;

    def __rmul__(self, other):
        return self * other;

    def tanh(self):
        _dataoutput = (math.exp(2 * self.data) - 1)/(math.exp(2*self.data) + 1)
        out = Value(_dataoutput, _children= (self, ), _op = 'tanh')
        def _backward():
            t = 1 - (_dataoutput)**2
            self.grad += t * out.grad
        out._backward = _backward
        return out;

    def backward(self):
        self.grad = 1
        visited = set()
        res = []
        def build_topo(node):
            if node in visited:
                return;
            visited.add(node)  
            for child in node._prev:
                if child not in visited:
                    build_topo(child)
            visited.remove(node)
            res.append(node)         

        build_topo(self)
        for v in reversed(res):
            v._backward();
        for v in res:
            print(v)



In [64]:
a = Value(2, label = 'a')
b = Value(-3, label = 'b')
c = Value(10, label = 'c')
f = Value(-2, label = 'f')
e = a * b
e.label = 'e'
d = e + c
d.label = 'd'
L = d * f
L.label = 'L'
L.backward()


Value(data=-2, label=f, gradient=4)
Value(data=10, label=c, gradient=-2)
Value(data=2, label=a, gradient=6)
Value(data=-3, label=b, gradient=-4)
Value(data=-6, label=e, gradient=-2)
Value(data=4, label=d, gradient=-2)
Value(data=-8, label=L, gradient=1)


In [65]:
a = Value(3.0)
b = a + a
b.backward()

Value(data=3.0, label=, gradient=2)
Value(data=6.0, label=, gradient=1)


In [66]:
x1 = Value(2, label = 'x1')
x2 = Value(0, label = 'x2')
w1 = Value(-3, label = 'w1')
w2 = Value(1, label = 'w2')
b = Value(6.8813735870195432, label = 'b')
x1w1 = x1 * w1
x1w1.label = 'x1*w1'
x2w2 = x2 * w2
x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = 'x1*w1 + x2*w2'
n =  x1w1 + x2w2 + b
n.label = 'x1*w1 + x2*w2 + b'
o = n.tanh()
o.backward()



Value(data=6.881373587019543, label=b, gradient=0.4999999999999999)
Value(data=0, label=x2, gradient=0.4999999999999999)
Value(data=1, label=w2, gradient=0.0)
Value(data=0, label=x2*w2, gradient=0.4999999999999999)
Value(data=-3, label=w1, gradient=0.9999999999999998)
Value(data=2, label=x1, gradient=-1.4999999999999996)
Value(data=-6, label=x1*w1, gradient=0.4999999999999999)
Value(data=-6, label=, gradient=0.4999999999999999)
Value(data=0.8813735870195432, label=x1*w1 + x2*w2 + b, gradient=0.4999999999999999)
Value(data=0.7071067811865476, label=, gradient=1)
